In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase

In [5]:
db = SQLDatabase.from_uri("sqlite:///../data/finance.db")

print(db.dialect)

print(db.get_usable_table_names())

sqlite
['accounts', 'customers', 'transactions']


In [6]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

chain = create_sql_query_chain(llm, db)

In [7]:
generated_sql_query = chain.invoke({"question": "고객의 이름을 나열하세요"})

print(generated_sql_query.__repr__())

'SELECT "name" \nFROM customers'


In [8]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

execute_query = QuerySQLDataBaseTool(db=db)

execute_query.invoke({"query": generated_sql_query})

C:\Users\user\AppData\Local\Temp\ipykernel_12308\3280318452.py:3: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  execute_query = QuerySQLDataBaseTool(db=db)


"[('테디',), ('폴',), ('셜리',), ('민수',), ('지영',), ('은정',)]"

In [9]:
write_query = create_sql_query_chain(llm, db)

chain = write_query | execute_query

In [10]:
chain.invoke({"question": "테디의 이메일을 조회하세요"})

"[('teddy@example.com',)]"

In [11]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer
    the user question.
    
    Question: {question}
    SQL Query: {query}
    SQL Result: {result}
    Answer: """
)

In [12]:
answer = answer_prompt | llm | StrOutputParser()

In [13]:
chain = (
    RunnablePassthrough.assign(query=write_query).assign(
        result=itemgetter("query") | execute_query
    )
    | answer
)

In [14]:
chain.invoke({"question": "테디의 transaction의 합계를 구하세요"})

'테디의 transaction의 합계는 -965.7 입니다.'

In [15]:
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

db = SQLDatabase.from_uri("sqlite:///../data/finance.db")

agent_executor = create_sql_agent(llm, db=db, agent_type="openai-tools", verbose=True)

In [16]:
agent_executor.invoke(
    {"input": "테디와 셜리의 transaction의 합계를 구하고 비교하세요"}
)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


accounts, customers, transactions
Invoking: `sql_db_schema` with `{'table_names': 'transactions'}`



CREATE TABLE transactions (
	transaction_id INTEGER, 
	account_id INTEGER, 
	amount REAL, 
	transaction_date TEXT, 
	PRIMARY KEY (transaction_id), 
	FOREIGN KEY(account_id) REFERENCES accounts (account_id)
)

/*
3 rows from transactions table:
transaction_id	account_id	amount	transaction_date
1	1	74.79	2024-07-13
2	1	-224.1	2024-05-13
3	1	-128.9	2024-01-25
*/
Invoking: `sql_db_query` with `{'query': 'SELECT account_id, SUM(amount) AS total_amount FROM transactions WHERE account_id IN (1, 2) GROUP BY account_id'}`


[(1, -965.7), (2, 743.13)]테디의 거래 합계는 -965.7이고, 셜리의 거래 합계는 743.13입니다.

> Finished chain.


{'input': '테디와 셜리의 transaction의 합계를 구하고 비교하세요',
 'output': '테디의 거래 합계는 -965.7이고, 셜리의 거래 합계는 743.13입니다.'}